In [ ]:
"""
protein_processing.py

Purpose:
Process BV-BRC E. coli genomes resistant to ciprofloxacin.
Revert known resistant mutations in gyrA/parC/parE to wild-type at DNA level.
Extract ALL CDS na_sequences from mutated/unmutated contigs, translate to proteins,
and save as multi-FASTA proteome files + summary CSV.

Key outputs:
- ecoli_pairs.csv : metadata + paths to mutated/unmutated protein FASTA files
- FASTA files in output directory: {genome_id}_mutated_proteins.faa and {genome_id}_unmutated_proteins.faa

Assumes all_cds_features.tsv from batched fetch (no na_sequence fetched; extracted here).
Run from command line: python protein_processing.py
"""

import pandas as pd
from pathlib import Path
from Bio.Seq import Seq
from Bio.Data import CodonTable
from tqdm.auto import tqdm
from rapidfuzz import fuzz
import os
from collections import Counter

In [3]:
# ────────────────────────────────────────────────
# CONFIGURATION
# ────────────────────────────────────────────────

CONTIGS_PATH    = "bvbrc_data/bvbrc_contigs.csv"
GENOMES_PATH    = "bvbrc_data/bvbrc_genomes.tsv"
GYRA_FEAT_PATH  = "bvbrc_data/gyrA_features.tsv"
PAR_FEAT_PATH   = "bvbrc_data/parC_parE_features.tsv"
CDS_FEAT_PATH   = "bvbrc_data/all_cds_features.tsv"
CARD_SNPS_PATH  = "card_data/cipro_snps_df.csv"

OUTPUT_DIR      = Path("ecoli_processed_pairs")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV      = OUTPUT_DIR / "ecoli_pairs.csv"
LOG_FILE        = OUTPUT_DIR / "MUT_LOGS.txt"

In [4]:
WT_AA_MAP = {
    'gyra': {
        83: 'S',   # Serine (S83L common resistance)
        87: 'D',   # Aspartic acid (D87N/G/Y common)
        # Add more: 81: 'G', 84: 'A', 106: 'H', etc.
    },
    'parc': {
        78: 'D',
        80: 'S',
        84: 'E',
    },
    'pare': {
        458: 'S',
        # Add more if known
    }
}

SIMILARITY_THRESHOLD = 99.0

In [5]:
# ────────────────────────────────────────────────
# STEP 1: Load codon tables for reversion
# ────────────────────────────────────────────────
print("[STEP 1] Loading standard genetic code for codon reversion...")
standard_table = CodonTable.unambiguous_dna_by_name["Standard"]
back_table = {}
for aa in 'ACDEFGHIKLMNPQRSTVWY*':
    back_table[aa] = []
for codon, aa in standard_table.forward_table.items():
    back_table[aa].append(codon)
for stop in standard_table.stop_codons:
    back_table['*'].append(stop)

print(f"  → Codon back-table created with {len(back_table)} amino acids")

[STEP 1] Loading standard genetic code for codon reversion...
  → Codon back-table created with 21 amino acids


In [6]:
# ────────────────────────────────────────────────
# STEP 2: Load all input data
# ────────────────────────────────────────────────
print("[STEP 2] Loading input files...")

contigs_df = pd.read_csv(CONTIGS_PATH, dtype={'genome_id': str})
contigs_df['contig_id'] = contigs_df['contig_id'].str.strip()
print(f"  → Loaded {len(contigs_df):,} contigs from {contigs_df['genome_id'].nunique()} genomes")

genomes_df = pd.read_csv(GENOMES_PATH, sep='\t', dtype={'genome_id': str})
print(f"  → Loaded metadata for {len(genomes_df)} genomes")

feature_cols = ['genome_id', 'patric_id', 'accession', 'start', 'end', 'strand', 'gene', 'product', 'na_sequence']
gyra_df = pd.read_csv(GYRA_FEAT_PATH, sep='\t', header=None, names=feature_cols, dtype={'genome_id': str, 'start': int, 'end': int})
par_df = pd.read_csv(PAR_FEAT_PATH, sep='\t', header=None, names=feature_cols, dtype={'genome_id': str, 'start': int, 'end': int})
features_df = pd.concat([gyra_df, par_df], ignore_index=True)
print(f"  → Loaded {len(features_df)} gyrA/parC/parE features from {features_df['genome_id'].nunique()} genomes")

cds_df = pd.read_csv(CDS_FEAT_PATH, sep='\t', dtype={'genome_id': str, 'start': int, 'end': int}, low_memory=False)
print(f"  → Loaded {len(cds_df):,} ALL CDS features from {cds_df['genome_id'].nunique()} genomes")

snps_df = pd.read_csv(CARD_SNPS_PATH)
snps_df['gene_key'] = snps_df['description'].str.lower().apply(
    lambda x: 'gyra' if 'gyra' in x else 'parc' if 'parc' in x else 'pare' if 'pare' in x else 'unknown'
)
snps_df = snps_df[snps_df['gene_key'] != 'unknown']
gene_to_snps = {k: snps_df[snps_df['gene_key'] == k].to_dict('records') for k in ['gyra', 'parc', 'pare']}
print(f"  → CARD SNPs loaded: {{k: len(v) for k,v in gene_to_snps.items()}}")

# Normalize contig_serial
features_df['accession'] = features_df['accession'].astype(str).str.strip()
features_df['contig_serial'] = features_df['genome_id'] + '.con.' + features_df['accession'].str[-4:]

cds_df['accession'] = cds_df['accession'].astype(str).str.strip()
cds_df['contig_serial'] = cds_df['genome_id'] + '.con.' + cds_df['accession'].str[-4:].astype(str)

features_df['gene_key'] = features_df['product'].str.lower().apply(
    lambda x: 'gyra' if 'gyrase subunit a' in x else
              'parc' if 'topoisomerase iv subunit a' in x else
              'pare' if 'topoisomerase iv subunit b' in x else 'unknown'
)
features_df = features_df[features_df['gene_key'] != 'unknown']

[STEP 2] Loading input files...
  → Loaded 73,423 contigs from 500 genomes
  → Loaded metadata for 500 genomes
  → Loaded 1669 gyrA/parC/parE features from 500 genomes
  → Loaded 2,583,020 ALL CDS features from 500 genomes
  → CARD SNPs loaded: {k: len(v) for k,v in gene_to_snps.items()}


In [7]:
features_df.head()

,genome_id,patric_id,accession,start,end,strand,gene,product,na_sequence,contig_serial,gene_key
0,562.8523,fig|562.8523.peg.3351,FBVS01000082,169917,172544,-,gyrA,DNA gyrase subunit A (EC 5.99.1.3),atgagcgaccttgcgagagaaattacaccggtcaacattgaggaag...,562.8523.con.0082,gyra
1,562.7542,fig|562.7542.peg.983,CXXY01000007,105310,107937,-,NaN,DNA gyrase subunit A (EC 5.99.1.3),atgagcgaccttgcgagagaaattacaccggtcaacattgaggaag...,562.7542.con.0007,gyra
2,562.98417,fig|562.98417.peg.691,562.98417.con.0002,258063,260690,-,NaN,DNA gyrase subunit A (EC 5.99.1.3),atgagcgaccttgcgagagaaattacaccggtcaacattgaggaag...,562.98417.con.0002,gyra
4,562.140842,fig|562.140842.peg.916,562.140842.con.0003,104287,106914,+,NaN,DNA gyrase subunit A (EC 5.99.1.3),atgagcgaccttgcgagagaaattacaccggtcaacattgaggaag...,562.140842.con.0003,gyra
5,562.135581,fig|562.135581.peg.114,562.135581.con.0001,117571,120198,+,NaN,DNA gyrase subunit A (EC 5.99.1.3),atgagcgaccttgcgagagaaattacaccggtcaacattgaggaag...,562.135581.con.0001,gyra


In [8]:
cds_df.head()

,genome_id,patric_id,accession,start,end,strand,gene,product,contig_serial
562.8523,562.8523,fig|562.8523.peg.2878,FBVS01000078,90517,92460,-,cpdB,"2',3'-cyclic-nucleotide 2'-phosphodiesterase (...",562.8523.con.0078
562.8523,562.8523,fig|562.8523.peg.2156,FBVS01000072,13530,14324,-,speD,S-adenosylmethionine decarboxylase proenzyme (...,562.8523.con.0072
562.8523,562.8523,fig|562.8523.peg.3666,FBVS01000090,56937,58220,-,gltA,Citrate synthase (si) (EC 2.3.3.1),562.8523.con.0090
562.8523,562.8523,fig|562.8523.peg.2389,FBVS01000076,115144,116070,-,yqhG,Uncharacterized protein YqhG,562.8523.con.0076
562.8523,562.8523,fig|562.8523.peg.1442,FBVS01000067,82403,82570,+,rmf,Ribosome modulation factor,562.8523.con.0067


In [9]:
# ────────────────────────────────────────────────
# Reversion helper
# ────────────────────────────────────────────────
def revert_mutations(na_sequence: str, gene_key: str, log_lines: list) -> tuple[str, bool, int]:
    aa_seq = str(Seq(na_sequence.upper()).translate())
    new_na = na_sequence.upper()
    reverted_count = 0

    for snp in gene_to_snps.get(gene_key, []):
        pos = snp['position'] - 1
        if pos >= len(aa_seq):
            continue
        current_aa = aa_seq[pos]
        if current_aa != snp['variant_aa']:
            continue

        wt_aa = WT_AA_MAP.get(gene_key, {}).get(snp['position'])
        if not wt_aa:
            continue

        dna_pos = pos * 3
        current_codon = new_na[dna_pos:dna_pos+3]
        possible_codons = back_table.get(wt_aa, [])
        reversion_codon = None

        for wc in possible_codons:
            diff = sum(a != b for a, b in zip(current_codon, wc))
            if diff == 1:
                reversion_codon = wc
                break

        if reversion_codon:
            new_na = new_na[:dna_pos] + reversion_codon + new_na[dna_pos+3:]
            reverted_count += 1
            log_lines.append(
                f"    Reverted {gene_key} pos {snp['position']} {current_aa}→{wt_aa} "
                f"({current_codon} → {reversion_codon})"
            )

    return new_na, reverted_count > 0, reverted_count

In [10]:
# ────────────────────────────────────────────────
# STEP 4: Process genomes
# ────────────────────────────────────────────────
print("\n[STEP 4] Processing genomes...")

pairs = []
log_lines = ["=== Mutation & Protein Extraction Log ===\n", f"Started: {pd.Timestamp.now()}\n\n"]

reversion_summary = Counter()
genomes_with_reversion = set()

genome_ids = genomes_df['genome_id'].unique()

for genome_id in tqdm(genome_ids, desc="Genomes"):
    log_lines.append(f"\n[GENOME] {genome_id}\n")

    genome_contigs = contigs_df[contigs_df['genome_id'] == genome_id].sort_values('fasta_order').copy()
    if genome_contigs.empty:
        log_lines.append("  Skipping: No contigs found\n")
        continue

    genome_features = features_df[features_df['genome_id'] == genome_id]
    unmut_contigs = genome_contigs.copy()
    genome_reverted_any = False
    genome_reversion_count = 0

    if genome_features.empty:
        log_lines.append("  No gyrA/parC/parE features → no reversions possible\n")
    else:
        for _, feat in genome_features.iterrows():
            contig_id = feat['contig_serial']
            if contig_id not in genome_contigs['contig_id'].values:
                log_lines.append(f"  Warning: No matching contig {contig_id} for feature\n")
                continue

            contig_row = genome_contigs[genome_contigs['contig_id'] == contig_id].iloc[0]
            contig_seq = contig_row['contig_sequence']

            start, end, strand = feat['start'], feat['end'], feat['strand']
            gene_slice = contig_seq[start-1:end]
            extracted = gene_slice if strand == '+' else str(Seq(gene_slice).reverse_complement())

            similarity = fuzz.ratio(extracted.upper(), feat['na_sequence'].upper())
            if similarity < SIMILARITY_THRESHOLD:
                log_lines.append(f"  Warning: Gene mismatch {feat['gene_key']} ({similarity:.1f}%) - skipping\n")
                continue

            log_lines.append(f"  Processing {feat['gene_key']} at {contig_id} [{start}-{end}] ({strand})\n")

            new_na_seq, reverted, count = revert_mutations(feat['na_sequence'], feat['gene_key'], log_lines)
            genome_reversion_count += count

            if reverted:
                genome_reverted_any = True
                genomes_with_reversion.add(genome_id)
                reversion_summary[feat['gene_key']] += count

                new_genomic = new_na_seq if strand == '+' else str(Seq(new_na_seq).reverse_complement())
                new_contig_seq = contig_seq[:start-1] + new_genomic + contig_seq[end:]

                unmut_contigs.loc[unmut_contigs['contig_id'] == contig_id, 'contig_sequence'] = new_contig_seq
                log_lines.append(f"  → {count} reversion(s) applied to {feat['gene_key']}\n")

        if not genome_reverted_any:
            log_lines.append("  No reversions applied\n")

    # ── Protein extraction ────────────────────────────────────────
    genome_cds = cds_df[cds_df['genome_id'] == genome_id]
    log_lines.append(f"  → {len(genome_cds)} CDS features found\n")

    mutated_proteins = []
    unmutated_proteins = []
    extracted_count = 0

    for _, cds_feat in genome_cds.iterrows():
        contig_serial = cds_feat['contig_serial']
        contig_mut_row = genome_contigs[genome_contigs['contig_id'] == contig_serial]
        contig_unmut_row = unmut_contigs[unmut_contigs['contig_id'] == contig_serial]

        if contig_mut_row.empty or contig_unmut_row.empty:
            if extracted_count < 5:  # limit spam
                log_lines.append(f"  No contig match for CDS {cds_feat.get('patric_id','?')} | serial={contig_serial}\n")
            continue

        seq_mut   = contig_mut_row.iloc[0]['contig_sequence']
        seq_unmut = contig_unmut_row.iloc[0]['contig_sequence']
        start, end, strand = cds_feat['start'], cds_feat['end'], cds_feat['strand']

        slice_mut   = seq_mut[start-1:end]
        na_mut      = slice_mut if strand == '+' else str(Seq(slice_mut).reverse_complement())

        slice_unmut = seq_unmut[start-1:end]
        na_unmut    = slice_unmut if strand == '+' else str(Seq(slice_unmut).reverse_complement())
        
        na_mut = na_mut[:len(na_mut) - (len(na_mut) % 3)]
        na_unmut = na_unmut[:len(na_unmut) - (len(na_unmut) % 3)]

        if len(na_mut) % 3 != 0 or len(na_unmut) % 3 != 0:
            log_lines.append(f"    Warning: trimmed {cds_feat.get('patric_id','?')} "
                             f"(mut: {len(na_mut)%3} bp, unmut: {len(na_unmut)%3} bp)\n")

        try:
            aa_mut   = str(Seq(na_mut).translate(to_stop=False))   # or table='Standard'
            aa_unmut = str(Seq(na_unmut).translate(to_stop=False))
        except Exception as e:
            log_lines.append(f"    Translation failed for {cds_feat.get('patric_id','?')}: {str(e)}\n")
            continue

        header = f">{cds_feat.get('patric_id', 'unknown')} | gene={cds_feat.get('gene', 'unknown')} | product={cds_feat.get('product', 'unknown')}"
        mutated_proteins.append(f"{header}\n{aa_mut}")
        unmutated_proteins.append(f"{header}\n{aa_unmut}")

        extracted_count += 1

        # Debug first few extractions
        if extracted_count <= 3:
            log_lines.append(f"    → Extracted CDS {extracted_count}: {len(aa_mut)} AA  ({cds_feat.get('product','?')[:40]}...)\n")

    log_lines.append(f"  Total proteins extracted: {extracted_count}\n")

    if extracted_count == 0:
        log_lines.append("  WARNING: ZERO proteins saved — check contig_serial matching or CDS data for this genome\n")

    # Save proteomes
    mut_faa   = OUTPUT_DIR / f"{genome_id}_mutated_proteins.faa"
    unmut_faa = OUTPUT_DIR / f"{genome_id}_unmutated_proteins.faa"

    with open(mut_faa, 'w') as f:
        if mutated_proteins:
            f.write('\n'.join(mutated_proteins) + '\n')
    with open(unmut_faa, 'w') as f:
        if unmutated_proteins:
            f.write('\n'.join(unmutated_proteins) + '\n')

    log_lines.append(
        f"  Saved mutated   proteome: {mut_faa} ({len(mutated_proteins):,} proteins)\n"
        f"  Saved unmutated proteome: {unmut_faa} ({len(unmutated_proteins):,} proteins)\n"
    )

    row = genomes_df[genomes_df['genome_id'] == genome_id].iloc[0].to_dict()
    row['mutated_protein_path']   = str(mut_faa)
    row['unmutated_protein_path'] = str(unmut_faa)
    row['reversions_applied']     = genome_reverted_any
    row['reversion_count']        = genome_reversion_count
    pairs.append(row)


[STEP 4] Processing genomes...


Genomes: 100%|██████████| 500/500 [18:18<00:00,  2.20s/it]


In [12]:
# ────────────────────────────────────────────────
# STEP 5: Save outputs + summary
# ────────────────────────────────────────────────
print("\n[STEP 5] Saving log, CSV and summary...")

with open(LOG_FILE, 'w', encoding='utf-8') as f:
    f.writelines(log_lines + ["\n=== END ===\n"])

print(f"  → Log saved to: {LOG_FILE}")

if pairs:
    pd.DataFrame(pairs).to_csv(OUTPUT_CSV, index=False)
    print(f"  → Saved {len(pairs)} entries to {OUTPUT_CSV}")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Processed {len(genome_ids)} genomes (limited mode)")
print(f"Genomes with reversions: {len(genomes_with_reversion)}")
print(f"Total reversions: {sum(reversion_summary.values())}")
for gene, cnt in sorted(reversion_summary.items(), key=lambda x: -x[1]):
    print(f"  • {gene.upper():<6} : {cnt:>4}")
print("="*60)
print("\nDone. Check MUT_LOGS.txt for details.\n")


[STEP 5] Saving log, CSV and summary...
  → Log saved to: ecoli_processed_pairs/MUT_LOGS.txt
  → Saved 500 entries to ecoli_processed_pairs/ecoli_pairs.csv

SUMMARY
Processed 500 genomes (limited mode)
Genomes with reversions: 382
Total reversions: 1081
  • GYRA   :  724
  • PARC   :  357

Done. Check MUT_LOGS.txt for details.



In [13]:
ecoli_pairs = pd.read_csv(OUTPUT_CSV)
ecoli_pairs.head()

,genome_id,antibiotic,resistant_phenotype,genome_name,taxon_id,taxon_lineage_names,genome_length,antimicrobial_resistance,antimicrobial_resistance_evidence,mutated_protein_path,unmutated_protein_path,reversions_applied,reversion_count
0,562.852300,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5235035,NaN,NaN,ecoli_processed_pairs/562.8523_mutated_protein...,ecoli_processed_pairs/562.8523_unmutated_prote...,True,3
1,562.754200,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5589103,NaN,NaN,ecoli_processed_pairs/562.7542_mutated_protein...,ecoli_processed_pairs/562.7542_unmutated_prote...,True,4
2,562.984170,ciprofloxacin,Resistant,Escherichia coli 99aee782-7bb9-11e9-a8d3-68b59...,562,cellular organisms::Bacteria::Pseudomonadati::...,5031625,Resistant::Susceptible,AMR Panel,ecoli_processed_pairs/562.98417_mutated_protei...,ecoli_processed_pairs/562.98417_unmutated_prot...,True,3
3,562.140842,ciprofloxacin,Resistant,Escherichia coli 018-E1-S06,562,cellular organisms::Bacteria::Pseudomonadati::...,4912847,Resistant::Intermediate::Susceptible,AMR Panel,ecoli_processed_pairs/562.140842_mutated_prote...,ecoli_processed_pairs/562.140842_unmutated_pro...,True,3
4,562.135581,ciprofloxacin,Resistant,Escherichia coli 20200203_MGL_06,562,cellular organisms::Bacteria::Pseudomonadati::...,5058347,NaN,NaN,ecoli_processed_pairs/562.135581_mutated_prote...,ecoli_processed_pairs/562.135581_unmutated_pro...,True,3


In [16]:
# get number of genome_ids where protein sequence length is 0
zero_protein_genomes = set()
for _, row in ecoli_pairs.iterrows():
    mut_path = row['mutated_protein_path']
    unmut_path = row['unmutated_protein_path']
    mut_count = 0
    unmut_count = 0

    if os.path.exists(mut_path):
        with open(mut_path) as f:
            mut_count = sum(1 for line in f if line.startswith('>'))
    if os.path.exists(unmut_path):
        with open(unmut_path) as f:
            unmut_count = sum(1 for line in f if line.startswith('>'))

    if mut_count == 0 or unmut_count == 0:
        zero_protein_genomes.add(row['genome_id'])
        print(f"Genome {row['genome_id']} has zero proteins in {'mutated' if mut_count == 0 else 'unmutated'} FASTA")
print(f"Genomes with zero proteins in either mutated or unmutated FASTA: {len(zero_protein_genomes)}")

Genome 1116003.3 has zero proteins in mutated FASTA
Genome 1116015.3 has zero proteins in mutated FASTA
Genome 1116020.3 has zero proteins in mutated FASTA
Genome 1116025.3 has zero proteins in mutated FASTA
Genome 1116103.3 has zero proteins in mutated FASTA
Genome 1116106.3 has zero proteins in mutated FASTA
Genome 1268975.3 has zero proteins in mutated FASTA
Genome 1268976.3 has zero proteins in mutated FASTA
Genome 1268977.3 has zero proteins in mutated FASTA
Genome 1268978.3 has zero proteins in mutated FASTA
Genome 1268979.3 has zero proteins in mutated FASTA
Genome 1268982.3 has zero proteins in mutated FASTA
Genome 1268983.3 has zero proteins in mutated FASTA
Genome 1268984.3 has zero proteins in mutated FASTA
Genome 1268985.3 has zero proteins in mutated FASTA
Genome 1268988.3 has zero proteins in mutated FASTA
Genome 1268994.3 has zero proteins in mutated FASTA
Genome 1268995.3 has zero proteins in mutated FASTA
Genome 1268997.3 has zero proteins in mutated FASTA
Genome 12689

In [ ]:
# drop records of sequences which have zero proteins in either mutated or unmutated FASTA
ecoli_pairs_filtered = ecoli_pairs[~ecoli_pairs['genome_id'].isin(zero_protein_genomes)]
print(f"After filtering, {len(ecoli_pairs_filtered)} records remain with valid protein sequences.")

After filtering, 469 records remain with valid protein sequences.


In [18]:
ecoli_pairs_filtered.to_csv(OUTPUT_CSV, index=False)
print(f"Filtered pairs saved to {OUTPUT_CSV}")

Filtered pairs saved to ecoli_processed_pairs/ecoli_pairs.csv


In [19]:
ecoli_pairs_filtered.head()

,genome_id,antibiotic,resistant_phenotype,genome_name,taxon_id,taxon_lineage_names,genome_length,antimicrobial_resistance,antimicrobial_resistance_evidence,mutated_protein_path,unmutated_protein_path,reversions_applied,reversion_count
0,562.852300,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5235035,NaN,NaN,ecoli_processed_pairs/562.8523_mutated_protein...,ecoli_processed_pairs/562.8523_unmutated_prote...,True,3
1,562.754200,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5589103,NaN,NaN,ecoli_processed_pairs/562.7542_mutated_protein...,ecoli_processed_pairs/562.7542_unmutated_prote...,True,4
2,562.984170,ciprofloxacin,Resistant,Escherichia coli 99aee782-7bb9-11e9-a8d3-68b59...,562,cellular organisms::Bacteria::Pseudomonadati::...,5031625,Resistant::Susceptible,AMR Panel,ecoli_processed_pairs/562.98417_mutated_protei...,ecoli_processed_pairs/562.98417_unmutated_prot...,True,3
3,562.140842,ciprofloxacin,Resistant,Escherichia coli 018-E1-S06,562,cellular organisms::Bacteria::Pseudomonadati::...,4912847,Resistant::Intermediate::Susceptible,AMR Panel,ecoli_processed_pairs/562.140842_mutated_prote...,ecoli_processed_pairs/562.140842_unmutated_pro...,True,3
4,562.135581,ciprofloxacin,Resistant,Escherichia coli 20200203_MGL_06,562,cellular organisms::Bacteria::Pseudomonadati::...,5058347,NaN,NaN,ecoli_processed_pairs/562.135581_mutated_prote...,ecoli_processed_pairs/562.135581_unmutated_pro...,True,3
